# <b>Исследование вероятности выполнения заявок логистической компанией

Логистическая компания занимается оказанием услуг по железнодорожной перевозке.  
Ежедневно в компанию поступают заявки на транспортировку товаров. Для каждой заявки известно направление перевозки (между какими станциями нужно осуществить транспортировку товаров), планируемая дата отправки, учетное направление перевозки (некоторые особенности перевозки). В течение жизненного цикла заявки (от ее регистрации в системе до её закрытия) с ней могут происходить различные изменения: изменение станций отправки и доставки, изменение величины прибыли, получаемой компанией в результате осуществления услуги по доставке, отмена заявки и т.д. 
В конце месяца из существующих заявок формируется заявочная компания, которая передается в отдел оптимизации, который составляет расписание движения составов, которое старается выполнить весь объем поступивших заявок.

**Глобальная задача:** используя инструменты машинного обучения предсказать вероятность выполнения заявки в заявочной компании. 

Вам дана информацию о том, как формировалась заявочная компания с 01.01.2023 по 30.09.2023. Для каждой заявки представлены все изменения, вносимые оператором и результат выполнения заявки (выполнение, перенос, отмена).

**Описание исходных данных:**
 -   `Идентификатор погрузки` - уникальный номер погрузки в системе данных компании;
 -   `Дата редактирования` - когда вносились изменения в информацию о заявке на выполнение погрузки;   
 -   `Версия погрузки` - номер редакции;
 -   `Оператор` -  человек, осуществляющий редактирование заявки;
 -   `Дата отправления` - планируемая дата отправки погрузки;        
 -   `Дата отправления факт` - фактическая дата отправки погрузки;
 -   `Учетное направление` - тип перевозки; 
 -   `Кластер назначения`- из какого региона двигается состав;     
 -   `Кластер отправления` -  в какой регион двигается состав; 
 -   `Код назначения` - станция откуда нужно перевести товары;           
 -   `Код отправления` - станция, куда нужно перевезти товары;         
 -   `Количество вагонов` - количество вагонов в составе;      
 -   `Контрибуция` - прибыль, которую получит компания за перевозку;                         
 -   `Идентификатор переноса` - была ли перенесена погрука (-1 – заявка отменена, 0 – заявка выполнена, 1 – заявка перенесена).                 
     

**Целевой признак**: статус выполнения заявки (1 -- если заявка выполнена, 0 -- если заявка не выполнена (перенесена, отменена)). 

*P.S.:* столбец целевого признака нужно создать самостоятельно.

## <b>Задания

### <b>Этап 1. Подготовка исходных данных

Подготовить исходные данные для решения задачи классификации:
1. Загрузите файл 'Orders_2023.xlsx;
2. Определите размер датасета.
3. Познакомьтесь с данными.
4. Проверьте типы данных.
5. Преобразуйте столбцы, содержащие дату и время к типу `datetime`. Убедитесь в том, что данные преобразовались адекватно. 
6. Проверьте пропуски, дубликаты. При выявлении пропусков, обработайте их.
7. Проверьте данные на артефакты.

Сформулируйте выводы.

In [4]:
import pandas as pd

In [5]:
df = pd.read_excel('Orders_2023.xlsx')

In [6]:
# смотрим размер датасета
df.shape

(18986, 14)

In [7]:
print(df.info())
df.head(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18986 entries, 0 to 18985
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Идентификатор погрузки  18986 non-null  int64  
 1   Дата редактирования     18884 non-null  object 
 2   версия погрузки         18986 non-null  int64  
 3   Оператор                18986 non-null  object 
 4   Дата отправления        18986 non-null  object 
 5   Дата отправления факт   13109 non-null  object 
 6   Учетное направление     18598 non-null  object 
 7   Кластер назначения      18898 non-null  object 
 8   Кластер отправления     18922 non-null  object 
 9   Код назначения          18986 non-null  int64  
 10  Код отправления         18986 non-null  int64  
 11  Количество вагонов      18986 non-null  int64  
 12  Контрибуция             18884 non-null  float64
 13  Идентификатор переноса  18986 non-null  int64  
dtypes: float64(1), int64(6), object(7)
mem

,Идентификатор погрузки,Дата редактирования,версия погрузки,Оператор,Дата отправления,Дата отправления факт,Учетное направление,Кластер назначения,Кластер отправления,Код назначения,Код отправления,Количество вагонов,Контрибуция,Идентификатор переноса
0,6832,13.12.2022 16:38,1,antoshkinaa,2023-01-16 00:00:00.000,NaN,(8) ВРП,ДВ,СПб,98220,3690,31,-1820795.0,-1
1,6832,26.12.2022 17:16,2,antoshkinaa,2023-01-16 00:00:00.000,NaN,(8) ВРП,ДВ,СПб,98220,3690,31,-1109524.0,-1
2,6832,16.01.2023 10:40,3,AntoshkinAA,2023-01-21 00:00:00.000,NaN,(8) ВРП,ДВ,СПб,98220,3690,31,-1109524.0,-1
3,6833,13.12.2022 17:36,1,antoshkinaa,2023-01-30 00:00:00.000,NaN,(1) Экспорт морской,ДВ,СПб,98220,3690,31,-1109524.0,-1
4,6833,26.12.2022 17:16,2,antoshkinaa,2023-01-30 00:00:00.000,NaN,(1) Экспорт морской,ДВ,СПб,98220,3690,31,-1109524.0,-1
5,6833,31.01.2023 07:50,3,antoshkinaa,2023-01-30 00:00:00.000,NaN,(1) Экспорт морской,ДВ,СПб,98220,3690,31,-1109524.0,-1
6,6833,01.02.2023 08:37,4,antoshkinaa,2023-01-30 00:00:00.000,NaN,(1) Экспорт морской,ДВ,СПб,98220,3690,31,-1109524.0,-1
7,6833,01.02.2023 08:37,5,antoshkinaa,2023-01-30 00:00:00.000,NaN,(1) Экспорт морской,ДВ,СПб,98220,3690,31,-1109524.0,-1
8,6840,2022-12-14 14:37:35.007,1,harinva,2023-01-15 00:00:00.000,2023-01-15 02:24:00.000,(8) ВРП,КРС,МСК,89210,23050,31,-394944.0,-1
9,6840,16.12.2022 13:29,2,harinva,2023-01-15 00:00:00.000,2023-01-15 02:24:00.000,(8) ВРП,КРС,МСК,89210,18290,31,-394944.0,-1


In [8]:
# создадим столбец целевого признака
df['Cтатус выполнения заявки'] = df['Идентификатор переноса'].map({
    -1: 0,
     0: 1,
     1: 0
})
df.head(1)

,Идентификатор погрузки,Дата редактирования,версия погрузки,Оператор,Дата отправления,Дата отправления факт,Учетное направление,Кластер назначения,Кластер отправления,Код назначения,Код отправления,Количество вагонов,Контрибуция,Идентификатор переноса,Cтатус выполнения заявки
0,6832,13.12.2022 16:38,1,antoshkinaa,2023-01-16 00:00:00.000,NaN,(8) ВРП,ДВ,СПб,98220,3690,31,-1820795.0,-1,0


In [9]:
# создадим словарь с кодами и кластерами, чтобы заменить неправильные и пустые значения
d = {}
for i in list(df['Код назначения'].dropna().unique()):
    # обработка ошибки, когда встречаются пустые массивы
    try:
        d.setdefault(i, df.query('`Код назначения` == @i')['Кластер назначения'].value_counts().reset_index().iloc[0, 0])
    except:
        continue

In [10]:
# повторим для второго столбца с кодами
for i in list(df['Код отправления'].dropna().unique()):
    # обработка ошибки, когда встречаются пуситые массивы
    try:
        d.setdefault(i, df.query('`Код отправления` == @i')['Кластер отправления'].value_counts().reset_index().iloc[0, 0])
    except:
        continue

In [11]:
a = set(list(df['Код отправления'].unique())+list(df['Код назначения'].unique()))

In [12]:
# получили коды назначения для которых нельзя определить кластер назначения
c = a - set(d.keys())
c

{1084,
 13470,
 22650,
 28090,
 28640,
 30180,
 31000,
 64840,
 65270,
 66640,
 72350,
 78590,
 80250,
 83266,
 84320,
 90230}

In [13]:
# получили всего 73 строк с этимим кодами в столбце код назначения
df.query('`Код назначения` in @c').shape

(73, 15)

In [14]:
# получили всего 62 строк с этимим кодами в столбце код отправления
df.query('`Код отправления` in @c').shape

(62, 15)

In [15]:
# с помощью созданного словаря заполнили заново кластеры
df['Кластер отправления'] = df['Код отправления'].map(d)
df['Кластер назначения'] = df['Код назначения'].map(d)

In [16]:
# исправили кластеры и заполнили пропусками там, где возможно
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18986 entries, 0 to 18985
Data columns (total 15 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Идентификатор погрузки    18986 non-null  int64  
 1   Дата редактирования       18884 non-null  object 
 2   версия погрузки           18986 non-null  int64  
 3   Оператор                  18986 non-null  object 
 4   Дата отправления          18986 non-null  object 
 5   Дата отправления факт     13109 non-null  object 
 6   Учетное направление       18598 non-null  object 
 7   Кластер назначения        18913 non-null  object 
 8   Кластер отправления       18924 non-null  object 
 9   Код назначения            18986 non-null  int64  
 10  Код отправления           18986 non-null  int64  
 11  Количество вагонов        18986 non-null  int64  
 12  Контрибуция               18884 non-null  float64
 13  Идентификатор переноса    18986 non-null  int64  
 14  Cтатус

In [17]:
df['Дата редактирования'] = pd.to_datetime(df['Дата редактирования'], format = 'mixed')
df['Дата отправления'] = pd.to_datetime(df['Дата отправления'])
df['Дата отправления факт'] = pd.to_datetime(df['Дата отправления факт'])

In [18]:
df = df.rename(columns = {'версия погрузки': 'Версия погрузки'})

In [19]:
# привели всех операторов к нижниму регистру и проверили на уникальность
df['Оператор'] = df['Оператор'].str.lower()
df['Оператор'].unique()

array(['antoshkinaa', 'harinva', 'rulskiya', 'pleshkovyi', 'gotselihya',
       'cvetaevaak', 'rodkevicha', 'elcovaiv', 'titovichsa', 'popovsi',
       'firsovaea', 'kuznecovaea', 'cimermanea', 'vasilevnd',
       'yurkevichev', 'novoselcevaaa', 'fesenkoe', 'ilichyovd',
       'suhovayv', 'babayanaa', 'matyushenkoav', 'gruzdevao',
       'samoylovichoa', 'blokovamv', 'chernokozinskiyln', 'volkovav',
       'popovaaal', 'belyaevaev', 'pahomovayv', 'nedayhlebas', 'devaevrd',
       'tsaplikmv', 'avdeevma', 'timofeevaas', 'jeleznayatn',
       'kivimyagiaa', 'automat', 'stepanovana', 'aleksandrma', 'mkom',
       'kirichkoa', 'yusupovaat'], dtype=object)

In [20]:
# исправили ошибку: более поздняя версия погрузки имеет более раннюю дату редактирования
df['Версия погрузки'] = df.groupby('Идентификатор погрузки')['Дата редактирования'].rank(method='first', ascending=True).astype('Int64')
df = df.sort_values(['Идентификатор погрузки', 'Версия погрузки'], ascending=[True, True])

In [21]:
# посмотрим, какую долю занимают наны в каждом из столбцов
df.isna().sum()/len(df)*100

Идентификатор погрузки       0.000000
Дата редактирования          0.537238
Версия погрузки              0.537238
Оператор                     0.000000
Дата отправления             0.000000
Дата отправления факт       30.954387
Учетное направление          2.043611
Кластер назначения           0.384494
Кластер отправления          0.326556
Код назначения               0.000000
Код отправления              0.000000
Количество вагонов           0.000000
Контрибуция                  0.537238
Идентификатор переноса       0.000000
Cтатус выполнения заявки     0.000000
dtype: float64

In [22]:
# посмотим количество дубликатов
df.duplicated().sum()

26

In [23]:
df.describe()

,Идентификатор погрузки,Дата редактирования,Версия погрузки,Дата отправления,Дата отправления факт,Код назначения,Код отправления,Количество вагонов,Контрибуция,Идентификатор переноса,Cтатус выполнения заявки
count,18986.000000,18884,18884.0,18986,13109,18986.000000,18986.000000,18986.000000,1.888400e+04,18986.000000,18986.000000
mean,9527.617402,2023-05-24 16:07:25.940175104,3.542099,2023-05-30 00:13:35.182766336,2023-06-03 07:21:55.541994240,60177.486885,68602.633098,27.393500,9.757058e+05,-0.577531,0.157326
min,6832.000000,2022-12-13 16:38:00,1.0,2023-01-02 00:00:00,2023-01-01 04:06:00,1037.000000,1037.000000,0.000000,-2.091830e+08,-1.000000,0.000000
25%,8204.250000,2023-03-07 07:50:00,2.0,2023-03-25 00:00:00,2023-04-03 10:04:00,23050.000000,23860.000000,31.000000,0.000000e+00,-1.000000,0.000000
50%,9605.000000,2023-05-30 17:32:30,3.0,2023-06-08 00:00:00,2023-06-12 14:41:00,78030.000000,85160.000000,31.000000,3.600000e+04,-1.000000,0.000000
75%,10770.000000,2023-07-31 14:58:45,5.0,2023-08-05 14:45:00,2023-08-09 20:42:00,89540.000000,95470.000000,31.000000,1.839503e+06,0.000000,0.000000
max,12672.000000,2023-12-09 17:22:00,38.0,2023-09-30 21:04:00,2023-09-30 23:52:00,99928.000000,98930.000000,76.000000,1.519000e+07,1.000000,1.000000
std,1530.478712,NaN,2.891711,NaN,NaN,36264.517816,33643.617526,14.306619,3.099752e+06,0.713553,0.364118


In [24]:
df['Учетное направление'].unique()

array(['(8) ВРП', '(1) Экспорт морской', nan, '(9) Подсыл порожних КТК',
       '(3) Экспорт сухопутный', '(4) Импорт морской',
       '(61) Иностранные территории', '(5) Импорт сухопутный',
       '(92) Подсыл порожних вагонов', '(2) Экспорт морской (повагонка)',
       '(6) Транзит', '(93) Досыл из под ПВ', '(7) Продажи НСК',
       '(91) Рефы'], dtype=object)

**Подумайте**: не содержит ли набор данных избыточной информации?

### Вывод
В результате первичного анализа исходного набора данных была проведена оценка его структуры, типов данных, наличия пропусков, дубликатов и возможных артефактов. Определён размер датасета, а также изучены представленные в нём признаки и их типы. Столбцы, содержащие сведения о датах и времени, были приведены к соответствующему типу datetime. Был сформирован столбец целевого признака, характеризующего статус выполнения заявки.

В ходе проверки качества данных были выявлены пропущенные значения. Было принято решение сохранить их без изменений. Также была проведена проверка на наличие дубликатов, по результатам которой было принято решение не удалять их, поскольку повторяющиеся записи могут отражать различные версии одной заявки.

В процессе анализа были выявлены ошибки в заполнении кластеров. Для их исправления был сформирован словарь соответствия, основанный на частоте использования уникального кода с определённым кластером. С его помощью были скорректированы значения кластеров. Были стандартизированы значения в столбце «Оператор», а также была выявлена и исправлена логическая ошибка в данных, при которой более поздней версии погрузки соответствовала более ранняя дата редактирования.

### <b>Этап 2. Создание новых признаков

8. Для исходного набора данных создайте новые признаки:
- `текущая длительность жизни заявки` - разница между датой появления заявки в системе и датой редактирования;
- `близость к отправке` - сколько времени осталось до плановой даты выполнения заявки;
- `периодичность редакции` - количество времени, прошедшее между редактированиями заявки - разница между текущей датой редактирования и предыдущей редакцией;
- `афтершоки` - количество дней, прошедшее после выполнения погрузки.
- `количество изменений до отправки` - количество раз, которое заявка редактировалась до отправки;
- `количество изменений после отправки` - сколько раз заявка редактировалась после того как была отправлена; 
- `количество переносов до отправки`; 
- `количество отмен до отправки`;
- `среднее количество времени`, прошедшее между редакциями;
- `длительность афтершока` - сколько времени прошло между датой последней редакции и осуществлением погрузки. 

In [25]:
# текущая длительность жизни заявки - разница между датой появления заявки в системе и датой редактирования
# Датой появления считаем первую дату редактирования заявки
df['Текущая длительность жизни заявки'] = (
    df['Дата редактирования']
    - df.groupby('Идентификатор погрузки')['Дата редактирования'].transform('min')
)

In [26]:
# близость к отправке - сколько времени осталось до плановой даты выполнения заявки
df['Близость к отправке'] = (
    df['Дата отправления'] - df['Дата редактирования']
)

In [27]:
# периодичность редакции - количество времени, прошедшее между редактированиями заявки -
# - разница между текущей датой редактирования и предыдущей редакцией
df['Периодичность редакции'] = df.groupby('Идентификатор погрузки')['Дата редактирования'].diff()

In [28]:
# афтершоки - количество дней, прошедшее после выполнения погрузки
df['Афтершоки'] = (
    df['Дата редактирования'] - df['Дата отправления факт']
).clip(lower=pd.Timedelta(0)).dt.days

In [29]:
# количество изменений до отправки - количество раз, которое заявка редактировалась до отправки
# Считаем накопительное количество редакций до фактической отправки
df['Количество изменений до отправки'] = (
    (df['Дата редактирования'] <= df['Дата отправления факт'])
    .groupby(df['Идентификатор погрузки'])
    .cumsum()
)

In [30]:
# количество изменений после отправки - сколько раз заявка редактировалась после того как была отправлена
# Считаем накопительное количество редакций после фактической отправки
df['Количество изменений после отправки'] = (
    (df['Дата редактирования'] > df['Дата отправления факт'])
    .groupby(df['Идентификатор погрузки'])
    .cumsum()
)

In [31]:
# количество переносов до отправки
# Считаем количество переносов до фактической отправки
df['Количество переносов до отправки'] = (
    (
        (df['Идентификатор переноса'] == 1) & (df['Дата редактирования'] <= df['Дата отправления факт'])
    )
    .groupby(df['Идентификатор погрузки'])
    .cumsum()
)

In [32]:
# количество отмен до отправки
# Считаем количество отмен до фактической отправки
df['Количество отмен до отправки'] = (
    (
        (df['Идентификатор переноса'] == -1) & (df['Дата редактирования'] <= df['Дата отправления факт'])
    )
    .groupby(df['Идентификатор погрузки'])
    .cumsum()
)

In [33]:
# среднее количество времени, прошедшее между редакциями
# Считаем среднее количество дней между редакциями каждой заявки
df['Среднее количество времени между редакциями'] = (
    df.groupby('Идентификатор погрузки')['Дата редактирования']
    .transform(lambda x: x.diff().mean())
)

In [34]:
# длительность афтершока - сколько времени прошло между датой последней редакции и осуществлением погрузки
# Считаем разницу между последней редакцией и фактической отправкой
df['Длительность афтершока'] = (
    df.groupby('Идентификатор погрузки')['Дата редактирования'].transform('max')
    - df['Дата отправления факт']
).clip(lower=pd.Timedelta(0))

In [35]:
df.head(8)

,Идентификатор погрузки,Дата редактирования,Версия погрузки,Оператор,Дата отправления,Дата отправления факт,Учетное направление,Кластер назначения,Кластер отправления,Код назначения,...,Текущая длительность жизни заявки,Близость к отправке,Периодичность редакции,Афтершоки,Количество изменений до отправки,Количество изменений после отправки,Количество переносов до отправки,Количество отмен до отправки,Среднее количество времени между редакциями,Длительность афтершока
0,6832,2022-12-13 16:38:00,1,antoshkinaa,2023-01-16,NaT,(8) ВРП,ДВ,СПб,98220,...,0 days 00:00:00,33 days 07:22:00,NaT,NaN,0,0,0,0,16 days 21:01:00,NaT
1,6832,2022-12-26 17:16:00,2,antoshkinaa,2023-01-16,NaT,(8) ВРП,ДВ,СПб,98220,...,13 days 00:38:00,20 days 06:44:00,13 days 00:38:00,NaN,0,0,0,0,16 days 21:01:00,NaT
2,6832,2023-01-16 10:40:00,3,antoshkinaa,2023-01-21,NaT,(8) ВРП,ДВ,СПб,98220,...,33 days 18:02:00,4 days 13:20:00,20 days 17:24:00,NaN,0,0,0,0,16 days 21:01:00,NaT
3,6833,2022-12-13 17:36:00,1,antoshkinaa,2023-01-30,NaT,(1) Экспорт морской,ДВ,СПб,98220,...,0 days 00:00:00,47 days 06:24:00,NaT,NaN,0,0,0,0,12 days 03:33:30,NaT
4,6833,2022-12-26 17:16:00,2,antoshkinaa,2023-01-30,NaT,(1) Экспорт морской,ДВ,СПб,98220,...,12 days 23:40:00,34 days 06:44:00,12 days 23:40:00,NaN,0,0,0,0,12 days 03:33:30,NaT
6,6833,2023-01-02 08:37:00,3,antoshkinaa,2023-01-30,NaT,(1) Экспорт морской,ДВ,СПб,98220,...,19 days 15:01:00,27 days 15:23:00,6 days 15:21:00,NaN,0,0,0,0,12 days 03:33:30,NaT
7,6833,2023-01-02 08:37:00,4,antoshkinaa,2023-01-30,NaT,(1) Экспорт морской,ДВ,СПб,98220,...,19 days 15:01:00,27 days 15:23:00,0 days 00:00:00,NaN,0,0,0,0,12 days 03:33:30,NaT
5,6833,2023-01-31 07:50:00,5,antoshkinaa,2023-01-30,NaT,(1) Экспорт морской,ДВ,СПб,98220,...,48 days 14:14:00,-2 days +16:10:00,28 days 23:13:00,NaN,0,0,0,0,12 days 03:33:30,NaT


### <b>Этап 3. Исследовательский анализ

Используя функцию ProfileReport библиотеки fg-data-profiling (https://github.com/Data-Centric-AI-Community/fg-data-profiling) провести первичный исследовательский анализ.

In [36]:
# pip install fg-data-profiling

In [37]:
from data_profiling import ProfileReport

In [38]:
# conda update typing_extensions

In [40]:
profile = ProfileReport(df, title="Profiling Report")

In [42]:
profile